# Sección 9: Violaciones en Contexto

Este notebook implementa el análisis de contexto temporal alrededor de las violaciones de
las dos invariantes de ProcessGuid detectadas en la sección 8.

**Archivos de entrada**
- `02_sysmon-run-01.csv` — CSV completo ordenado cronológicamente
- `04_sysmon-run-01-violations.csv` — eventos que participan en alguna violación
- `04_processguid-pid-violations-run-01.csv` — GUIDs violadores de Invariante 1
- `04_processguid-image-violations-run-01.csv` — GUIDs violadores de Invariante 2

---

## Paso 1: Importaciones y constantes

In [ ]:
import pandas as pd
import numpy as np
import warnings

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.max_colwidth', 80)
warnings.filterwarnings('ignore')

NULL_GUID = '00000000-0000-0000-0000-000000000000'
WINDOW    = 15   # eventos antes y después de cada violación

DATA_DIR = '../dataset/run-01-apt-1/'

# k-pairs definidos en la sección 8g
PAIRS = [
    (1, 'ProcessGuid',       'ProcessId',       'Image',       'eid_not_8_10'),
    (2, 'ParentProcessGuid', 'ParentProcessId', 'ParentImage', 'eid_eq_1'),
    (3, 'SourceProcessGUID', 'SourceProcessId', 'SourceImage', 'eid_8_10'),
    (4, 'TargetProcessGUID', 'TargetProcessId', 'TargetImage', 'eid_8_10'),
]
DOMAIN_FN = {
    'eid_not_8_10': lambda d: d[~d['EventID'].isin([8, 10])],
    'eid_eq_1':     lambda d: d[d['EventID'] == 1],
    'eid_8_10':     lambda d: d[d['EventID'].isin([8, 10])],
}

print('Constantes cargadas')

## Paso 2: Carga de datos

In [ ]:
# CSV completo — base para las ventanas de contexto
df = pd.read_csv(DATA_DIR + '02_sysmon-run-01.csv', low_memory=False)
df['_original_row_index'] = df.index
# Filtrar timestamps sentinela negativos antes de parsear
df['ts'] = pd.to_datetime(df['timestamp'].where(df['timestamp'] > 0), unit='ms')

print(f'df cargado: {len(df):,} filas  |  {df.columns.size} columnas')
print(f'Rango temporal: {df["ts"].min()} → {df["ts"].max()}')

In [ ]:
# CSV de eventos-violación
viol = pd.read_csv(DATA_DIR + '04_sysmon-run-01-violations.csv', low_memory=False)
viol['ts'] = pd.to_datetime(viol['timestamp'].where(viol['timestamp'] > 0), unit='ms')

print(f'viol cargado: {len(viol):,} eventos')

# Catálogos de violaciones de los dos invariantes
pid_viol = pd.read_csv(DATA_DIR + '04_processguid-pid-violations-run-01.csv')
img_viol = pd.read_csv(DATA_DIR + '04_processguid-image-violations-run-01.csv')

print(f'pid_viol: {len(pid_viol)} filas  |  img_viol: {len(img_viol)} filas')

---
## Invariante 1: GUID centinela en k=1

El GUID centinela `00000000-0000-0000-0000-000000000000` viola el Invariante 1
en k=1 con **36 eventos y 14 PIDs distintos**.

In [ ]:
# Eventos centinela en k=1 (dominio: EID ∉ {8,10}), ordenados por posición en CSV
sentinel_k1 = (
    viol[
        ~viol['EventID'].isin([8, 10]) &
        (viol['ProcessGuid'] == NULL_GUID)
    ]
    .sort_values('_original_row_index')
)

print(f'Eventos centinela en k=1 : {len(sentinel_k1)}')
print(f'PIDs distintos           : {sentinel_k1["ProcessId"].nunique()}')
print(f'Rango temporal           : {sentinel_k1["ts"].min()} → {sentinel_k1["ts"].max()}')

sentinel_k1[['ts', 'EventID', 'ProcessGuid', 'ProcessId', 'Image', 'Computer']].head(10)

### Ventana de contexto: primer evento centinela (k=1)

In [ ]:
first = sentinel_k1.iloc[0]
idx   = int(first['_original_row_index'])

window = df.iloc[max(0, idx - WINDOW) : idx + WINDOW + 1].copy()
window['▶'] = ''
window.loc[idx, '▶'] = '◀ violación'

cols = ['▶', 'ts', 'EventID', 'ProcessGuid', 'ProcessId', 'Image', 'Computer']
window[cols].reset_index(drop=True)

**¿Qué observar?**
- Los eventos inmediatamente anteriores tienen GUIDs reales — el driver ya estaba funcionando.
- El evento centinela interrumpe la secuencia: Sysmon registra el evento pero no tiene GUID disponible.
- Los eventos posteriores retoman GUIDs reales: el driver recuperó visibilidad.

---
## Invariante 2: Colisión genuina (svchost / dxgiadaptercache)

Dos GUIDs donde `svchost.exe` y `dxgiadaptercache.exe` comparten ProcessGuid.
**119 eventos** en total, ambos con ProcessId=1868.

In [ ]:
genuine_guid_1 = '2d5a9c51-5053-67da-2000-000000009000'
genuine_guid_2 = '2d5a9c51-505c-67da-2500-000000009000'

# Ciclo de vida completo del primer GUID violador (dominio k=1: EID ∉ {8,10})
guid_events = (
    df[
        ~df['EventID'].isin([8, 10]) &
        df['ProcessGuid'].isin([genuine_guid_1])
    ]
    .copy()
    .sort_values('ts')
)

print(f'Total eventos para {genuine_guid_1}: {len(guid_events)}')
print(f'\nDistribución por Image:')
print(guid_events['Image'].value_counts())
print(f'\nDistribución por EventID:')
print(guid_events['EventID'].value_counts().sort_index())

cols = ['ts', 'EventID', 'ProcessGuid', 'ProcessId', 'Image', 'Computer']
guid_events[cols]

### Ventana alrededor del cambio de Image

In [ ]:
dominant_image = guid_events['Image'].mode()[0]
change_rows    = guid_events[guid_events['Image'] != dominant_image]

if len(change_rows) > 0:
    change_idx = int(change_rows.iloc[0]['_original_row_index'])
    window = df.iloc[max(0, change_idx - WINDOW) : change_idx + WINDOW + 1].copy()
    window['▶'] = ''
    window.loc[change_idx, '▶'] = '◀ Image cambia aquí'
    display(
        window[['▶', 'ts', 'EventID', 'ProcessGuid', 'ProcessId', 'Image', 'Computer']]
        .reset_index(drop=True)
    )
else:
    print('No se encontró cambio de Image en el dominio k=1 para este GUID')

**¿Qué observar?**
- ¿Comparten el mismo `ProcessId` antes y después del cambio?
- Si el PID cambia junto con la Image → dos procesos distintos colisionaron en el mismo GUID.
- Si el PID es idéntico → posible reuso del GUID por un proceso sucesor.

---
## Invariante 2: Artefacto `<unknown process>`

17 GUIDs con eventos iniciales registrados como `<unknown process>` antes de que
Sysmon tuviera visibilidad del ejecutable real.

In [ ]:
unknown_guids = img_viol[img_viol['Image'] == '<unknown process>']['ProcessGuid'].unique()
guid_unknown  = unknown_guids[0]

guid_ev = (
    df[
        ~df['EventID'].isin([8, 10]) &
        (df['ProcessGuid'] == guid_unknown)
    ]
    .copy()
    .sort_values('ts')
)

print(f'GUID: {guid_unknown}')
print(f'\nDistribución de Image:')
print(guid_ev['Image'].value_counts())
print(f'\nCronología (primeros 10 eventos):')
guid_ev[['ts', 'EventID', 'Image', 'ProcessId', 'Computer']].head(10)

El patrón es consistente: pocos eventos iniciales con `<unknown process>`,
seguidos por todos los eventos restantes con la Image real.
Esto confirma que la regla de corrección automática (reemplazar por la Image dominante) es segura.

---
## Actividad Práctica

### Ejercicio A — Centinela en k=2

El centinela acumula **500 eventos** en k=2 (`ParentProcessGuid`).
Adapta el código de ventana de contexto para el primer evento centinela de k=2 y responde:
- ¿Qué EventID tienen estos eventos?
- ¿Cuántos procesos distintos (por `Image`) tienen al centinela como padre?
- ¿En qué ventana temporal ocurren la mayoría?

In [ ]:
# --- Ejercicio A: completa el código ---

# k=2 usa dominio EID=1 y la columna ParentProcessGuid
sentinel_k2 = (
    viol[
        (viol['EventID'] == 1) &
        (viol['ParentProcessGuid'] == NULL_GUID)
    ]
    .sort_values('_original_row_index')
)

print(f'Eventos centinela en k=2 : {len(sentinel_k2)}')

# TODO: mostrar distribución de EventID
# TODO: contar Images distintas (columna Image = proceso hijo)
# TODO: mostrar ventana de contexto del primer evento centinela k=2

### Ejercicio B — Inspección manual de la colisión genuina

Para los dos GUIDs con colisión genuina (`genuine_guid_1`, `genuine_guid_2`):
1. ¿El `ProcessId` es el mismo en todos los eventos, o cambia cuando cambia la `Image`?
2. ¿Qué EventIDs están presentes antes y después del cambio de Image?
3. ¿Se trata de un único proceso o de dos procesos distintos que colisionaron en el mismo GUID?
4. ¿Qué columnas adicionales (`ParentProcessGuid`, `ParentImage`, `CommandLine`) ayudarían a decidir?

In [ ]:
# --- Ejercicio B: completa el análisis ---

# Eventos del segundo GUID violador
guid_events_2 = (
    df[
        ~df['EventID'].isin([8, 10]) &
        df['ProcessGuid'].isin([genuine_guid_2])
    ]
    .copy()
    .sort_values('ts')
)

print(f'Eventos para {genuine_guid_2}: {len(guid_events_2)}')
print(guid_events_2['Image'].value_counts())

# TODO: verificar si ProcessId cambia junto con Image
# TODO: comparar EventIDs antes y después del cambio
# TODO: mostrar columnas ParentProcessGuid, ParentImage, CommandLine para los eventos del cambio

### Ejercicio C — Función de corrección para `<unknown process>`

In [ ]:
def fix_unknown_process(df, guid):
    """
    Reemplaza '<unknown process>' en Image con la Image dominante del GUID.
    Retorna el número de filas corregidas.
    """
    mask   = (~df['EventID'].isin([8, 10])) & (df['ProcessGuid'] == guid)
    events = df[mask]

    real_images = events[events['Image'] != '<unknown process>']['Image']
    if real_images.empty:
        return 0

    dominant     = real_images.mode()[0]
    unknown_mask = mask & (df['Image'] == '<unknown process>')
    df.loc[unknown_mask, 'Image'] = dominant
    return int(unknown_mask.sum())


# Prueba con el primer GUID de la categoría <unknown process>
df_copy = df.copy()   # trabajar sobre copia para no mutar df original
n_fixed = fix_unknown_process(df_copy, guid_unknown)

print(f'Filas corregidas: {n_fixed}')
print(df_copy[~df_copy['EventID'].isin([8, 10]) & (df_copy['ProcessGuid'] == guid_unknown)]['Image'].value_counts())

# TODO: ¿qué ocurre si un GUID tiene '<unknown process>' en TODOS sus eventos?
# Prueba ese caso límite aquí: